In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. LOAD DATA & CLEAN DURATION ANOMALIES
# ==============================================================================
df_features = pd.read_csv(
    r"C:\Users\jrdie\OneDrive\Desktop\BTT\kidney_waitlist_analytic_opt.csv",
    low_memory=False,
)

# Filter out the 59 corrupt records with missing or negative days_to_event
df_features = df_features[df_features["days_to_event"] > 0].copy()

# ==============================================================================
# 2. CLINICAL DECODING: FUNC_STAT_TCR (Per data_dictionary.md)
# ==============================================================================
# Adult Karnofsky: 2010 - 2100 -> 10% - 100%
# Pediatric Lansky: 4010 - 4100 -> 10% - 100%
# Code 1.0: ADL independent -> 100%
# Codes 996, 998, -1.0, or NaN: categorical non-scale or unknown
scale_digit = df_features["FUNC_STAT_TCR"] // 1000

conditions = [
    scale_digit == 2,
    scale_digit == 4,
    df_features["FUNC_STAT_TCR"] == 1.0,
    df_features["FUNC_STAT_TCR"] == 996.0,
    df_features["FUNC_STAT_TCR"] == 998.0,
]

choices = [
    "karnofsky",
    "lansky",
    "adl_independent",
    "not_applicable_infant",
    "unknown",
]

df_features["functional_scale"] = np.select(conditions, choices, default="unknown")

# Extract functional percentage (2080 -> 80.0, 4100 -> 100.0, 1.0 -> 100.0)
is_scale = (scale_digit == 2) | (scale_digit == 4)
df_features["functional_percent"] = np.where(
    is_scale,
    df_features["FUNC_STAT_TCR"] % 1000,
    np.where(df_features["FUNC_STAT_TCR"] == 1.0, 100.0, np.nan),
)

# Impute non-scale / missing with median and track via indicator
df_features["functional_percent_missing"] = (
    df_features["functional_percent"].isna().astype(int)
)
df_features["functional_percent"] = df_features["functional_percent"].fillna(
    df_features["functional_percent"].median()
)

# ==============================================================================
# 3. DATE-BASED DERIVATIONS & OUTLIER MITIGATION
# ==============================================================================
# 1. Dialysis Vintage / Duration Handling
if "DIALYSIS_DURATION" in df_features.columns:
  # DIALYSIS_DURATION was already engineered and saved in this CSV
  df_features["DIALYSIS_DURATION"] = (
      df_features["DIALYSIS_DURATION"].clip(lower=0).fillna(0)
  )
elif "DIALYSIS_DATE" in df_features.columns and "INIT_DATE" in df_features.columns:
  init_dt = pd.to_datetime(df_features["INIT_DATE"], errors="coerce")
  dialysis_dt = pd.to_datetime(df_features["DIALYSIS_DATE"], errors="coerce")
  raw_vintage = (init_dt - dialysis_dt).dt.days
  df_features["DIALYSIS_DURATION"] = raw_vintage.clip(lower=0).fillna(0)
else:
  # Fallback if neither exists
  df_features["DIALYSIS_DURATION"] = 0.0

# Indicator if dialysis occurred after listing
if "DIALYSIS_AFTER_LISTING" not in df_features.columns:
  if "DIALYSIS_DATE" in df_features.columns:
    df_features["DIALYSIS_AFTER_LISTING"] = (
        (
            pd.to_datetime(df_features["INIT_DATE"], errors="coerce")
            - pd.to_datetime(df_features["DIALYSIS_DATE"], errors="coerce")
        ).dt.days
        < 0
    ).astype(int)
  else:
    df_features["DIALYSIS_AFTER_LISTING"] = 0

# Non-linear log transform for dialysis vintage
df_features["DIALYSIS_DURATION_LOG"] = np.log1p(
    df_features["DIALYSIS_DURATION"]
)

# 2. Policy Era Flag (March 15, 2021 Acuity Circles)
if "INIT_DATE" in df_features.columns:
  init_dt = pd.to_datetime(df_features["INIT_DATE"], errors="coerce")
  df_features["LISTED_POST_2021"] = (
      init_dt >= pd.to_datetime("2021-03-15")
  ).astype(int)
elif "LISTED_POST_2021" not in df_features.columns:
  df_features["LISTED_POST_2021"] = 0

# 3. Clinical interaction: Age x Years on Dialysis
df_features["AGE_DIALYSIS_INTERACTION"] = df_features["INIT_AGE"] * (
    df_features["DIALYSIS_DURATION"] / 365.25
)

# 4. Outlier mitigation and null imputations
if "BMI_TCR" in df_features.columns:
  df_features.loc[df_features["BMI_TCR"] > 80.0, "BMI_TCR"] = np.nan
  df_features["BMI_TCR"] = df_features["BMI_TCR"].fillna(
      df_features["BMI_TCR"].median()
  )

if "INIT_CPRA" in df_features.columns:
  df_features["INIT_CPRA"] = df_features["INIT_CPRA"].fillna(0.0)

if "A2A2B_ELIGIBILITY" in df_features.columns:
  df_features["A2A2B_ELIGIBILITY"] = (
      df_features["A2A2B_ELIGIBILITY"].fillna(-1).astype(str)
  )

# ==============================================================================
# 4. FEATURE SELECTION (EXPLICIT DAY-1 WHITELIST)
# ==============================================================================
categorical_cols = [
    "ON_DIALYSIS",
    "GENDER",
    "ABO",
    "A2A2B_ELIGIBILITY",
    "INIT_STAT",
    "ETHCAT",
    "REGION",
    "functional_scale",
]

numerical_cols = [
    "BMI_TCR",
    "INIT_CPRA",
    "INIT_AGE",
    "functional_percent",
    "DIALYSIS_DURATION",
    "DIALYSIS_DURATION_LOG",
    "AGE_DIALYSIS_INTERACTION",
]

binary_indicator_cols = [
    "DIALYSIS_AFTER_LISTING",
    "LISTED_POST_2021",
    "functional_percent_missing",
]

target_cols = [
    "outcome",
    "event_adverse",
    "event_transplant",
    "censored",
    "days_to_event",
]

# Build analytic frame
df_model = df_features[
    categorical_cols + numerical_cols + binary_indicator_cols + target_cols
].copy()

# ==============================================================================
# 5. ENCODING, SCALING & EXPORT
# ==============================================================================
# One-Hot Encode categoricals with dummy drop
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

# Standardize continuous variables
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

# Verification checks
print(f"Final analytic matrix shape: {df_encoded.shape}")
print(
    "Total null values in features:"
    f" {df_encoded.drop(columns=target_cols).isna().sum().sum()}"
)

# Ready to save:
# df_encoded.to_csv("../data/processed/task3_encoded_scaled.csv", index=False)

Final analytic matrix shape: (494013, 51)
Total null values in features: 0
